# 🔬 XAS Data Visualization & Analysis Agent

Interactive notebook for visualizing and analyzing X-ray Absorption Spectroscopy (XAS) data.

**Workflow:**
1. Load all scans from `731_Data/`
2. Browse scan summary table
3. Select a scan and signal type (TEY, TFY, MCP) to plot
4. Optionally normalize by I0
5. Compare multiple scans
6. Export selected data to `exported_data/`

In [ ]:
# ── Imports & Setup ──────────────────────────────────────────────────────────
import os, sys, datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import ipywidgets as widgets
from IPython.display import display, clear_output

# Make sure our utility module is importable
sys.path.insert(0, os.path.abspath("."))
import xas_utils as xu

# Plotting defaults
matplotlib.rcParams.update({
    "figure.figsize": (10, 6),
    "font.size": 12,
    "axes.grid": True,
    "grid.alpha": 0.3,
})
%matplotlib inline

print("✅ Imports complete.")

## 1 · Load All Scans

In [ ]:
scans = xu.load_all_scans("731_Data")
print(f"Loaded {len(scans)} scans.\n")
summary = xu.scan_summary(scans)
display(summary)

## 2 · Interactive Single-Scan Plotter

Select a scan and a signal channel, then click **Plot** to visualize.

In [ ]:
# ── Widget definitions ────────────────────────────────────────────────────────
scan_ids = list(scans.keys())

w_scan = widgets.Dropdown(options=scan_ids, description="Scan:")
w_signal = widgets.Dropdown(options=["TEY", "TFY", "MCP"], value="TEY", description="Signal:")
w_norm = widgets.Checkbox(value=False, description="Normalize by I0")
w_plot_btn = widgets.Button(description="Plot", button_style="primary", icon="chart-line")
w_save_btn = widgets.Button(description="Save Data", button_style="success", icon="save")
w_filename = widgets.Text(value="", placeholder="optional filename (auto-generated if blank)", description="Filename:")
out_plot = widgets.Output()
out_status = widgets.Output()

# State to hold the last plotted data for saving
_last_plot = {}

def _on_plot(btn):
    """Plot the selected scan / signal."""
    with out_plot:
        clear_output(wait=True)
        sid = w_scan.value
        sig = w_signal.value
        df = scans[sid]["df"]
        meta = scans[sid]["meta"]
        energy = xu.get_energy(df)

        if w_norm.value:
            signal = xu.normalize_by_i0(df, sig)
            ylabel = f"{sig} / I0"
        else:
            signal = xu.get_signal(df, sig)
            ylabel = sig

        fig, ax = plt.subplots()
        ax.plot(energy, signal, "b-", linewidth=1.2)
        ax.set_xlabel("Mono Energy (eV)")
        ax.set_ylabel(ylabel)
        ax.set_title(f"{sid}  —  {meta['scan_type']}  ({meta['date']})")
        plt.tight_layout()
        plt.show()

        # Store for potential save
        _last_plot["energy"] = energy
        _last_plot["signal"] = signal
        _last_plot["signal_name"] = ylabel
        _last_plot["scan_id"] = sid

    with out_status:
        clear_output()
        print(f"Plotted {sig} for {sid}  |  E range: {energy.min():.2f} – {energy.max():.2f} eV  |  {len(energy)} points")


def _on_save(btn):
    """Save the last plotted data to an ASCII file."""
    with out_status:
        clear_output()
        if not _last_plot:
            print("⚠️  Nothing to save — plot something first.")
            return
        fname = w_filename.value.strip() or None
        path = xu.export_data(
            energy=_last_plot["energy"],
            signal=_last_plot["signal"],
            signal_name=_last_plot["signal_name"],
            scan_id=_last_plot["scan_id"],
            filename=fname,
        )
        print(f"💾 Saved → {path}")

w_plot_btn.on_click(_on_plot)
w_save_btn.on_click(_on_save)

# ── Layout ─────────────────────────────────────────────────────────────────────
controls = widgets.VBox([
    widgets.HBox([w_scan, w_signal, w_norm]),
    widgets.HBox([w_plot_btn, w_save_btn, w_filename]),
])
display(controls, out_plot, out_status)

## 3 · Multi-Scan Comparison

Select multiple scans and overlay them on a single plot.

In [ ]:
w_multi = widgets.SelectMultiple(
    options=scan_ids,
    description="Scans:",
    rows=min(10, len(scan_ids)),
)
w_sig_multi = widgets.Dropdown(options=["TEY", "TFY", "MCP"], value="TEY", description="Signal:")
w_norm_multi = widgets.Checkbox(value=False, description="Normalize by I0")
w_compare_btn = widgets.Button(description="Compare", button_style="info", icon="layer-group")
out_compare = widgets.Output()

def _on_compare(btn):
    with out_compare:
        clear_output(wait=True)
        selected = list(w_multi.value)
        if not selected:
            print("⚠️  Select at least one scan.")
            return
        sig = w_sig_multi.value
        fig, ax = plt.subplots()
        for sid in selected:
            df = scans[sid]["df"]
            energy = xu.get_energy(df)
            if w_norm_multi.value:
                signal = xu.normalize_by_i0(df, sig)
                ylabel = f"{sig} / I0"
            else:
                signal = xu.get_signal(df, sig)
                ylabel = sig
            ax.plot(energy, signal, linewidth=1.0, label=sid)
        ax.set_xlabel("Mono Energy (eV)")
        ax.set_ylabel(ylabel)
        ax.set_title(f"Comparison — {sig}")
        ax.legend(fontsize=9, ncol=2)
        plt.tight_layout()
        plt.show()

w_compare_btn.on_click(_on_compare)

display(
    widgets.HBox([w_multi, widgets.VBox([w_sig_multi, w_norm_multi, w_compare_btn])]),
    out_compare,
)

## 4 · Quick Plot (Code Cell)

For quick scripting — change the variables below and run the cell.

In [ ]:
# ── Quick-plot parameters (edit these) ────────────────────────────────────────
SCAN_ID   = "SigScan45611"   # change to any loaded scan
SIGNAL    = "TEY"            # TEY | TFY | MCP
NORM_I0   = False            # True to divide by I0

# ── Plot ───────────────────────────────────────────────────────────────────────
df = scans[SCAN_ID]["df"]
meta = scans[SCAN_ID]["meta"]
energy = xu.get_energy(df)
signal = xu.normalize_by_i0(df, SIGNAL) if NORM_I0 else xu.get_signal(df, SIGNAL)
label = f"{SIGNAL}/I0" if NORM_I0 else SIGNAL

fig, ax = plt.subplots()
ax.plot(energy, signal, "b-", lw=1.2)
ax.set_xlabel("Mono Energy (eV)")
ax.set_ylabel(label)
ax.set_title(f"{SCAN_ID} — {meta['scan_type']} ({meta['date']})")
plt.tight_layout()
plt.show()

## 5 · Export Current Plot Data

Run this cell after a quick-plot to save the data. You will be prompted for a filename.

In [ ]:
# Provide a filename or leave as None for auto-generated name
EXPORT_FILENAME = None   # e.g. "my_export.txt"

path = xu.export_data(
    energy=energy,
    signal=signal,
    signal_name=label,
    scan_id=SCAN_ID,
    filename=EXPORT_FILENAME,
)
print(f"💾 Exported to: {path}")

## 6 · Inspect Raw Data

View the raw DataFrame and metadata for any scan.

In [ ]:
INSPECT_SCAN = "SigScan45611"

print("=== Metadata ===")
for k, v in scans[INSPECT_SCAN]["meta"].items():
    print(f"  {k}: {v}")

print("\n=== Column Names ===")
print(list(scans[INSPECT_SCAN]["df"].columns))

print("\n=== First 5 rows ===")
display(scans[INSPECT_SCAN]["df"].head())